# Nhận diện tin giả đa phương tiện (Two-Stream Fusion) – Colab
BERT + ResNet-50 + chú ý chéo + Dense/Softmax. Nhãn: **0 = Fake, 1 = Real**.

**Trước khi chạy:** *Runtime → Change runtime type → T4 GPU*. Chạy các ô từ trên xuống, từng mục một.

## 1. Kiểm tra GPU và cài thư viện

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU (huấn luyện sẽ rất chậm)')

In [ ]:
!pip -q install transformers scikit-learn pandas pillow

## 2. Ghi các file mã nguồn

In [ ]:
%%writefile model.py
"""Mô hình Two-Stream Fusion cho nhận diện tin giả đa phương tiện (tiểu luận, mục 3.4).

Luồng dữ liệu (B = batch, N = số token, d = 768 với bert-base):
  văn bản  --BERT-->            T      [B, N, d]
  ảnh      --ResNet-50 + Linear--> V   [B, 49, d]      (bước 2)
  Cross-Attention: Q = T, K = V, Val = V -> H [B, N, d]   (bước 3)
  z = [t_CLS ; MeanPool(H)]                [B, 2d]
  Dense -> ReLU -> Dense -> logits         [B, 2]       (bước 4; softmax nằm trong CrossEntropyLoss)
Nhãn: 0 = Fake, 1 = Real.
"""
import torch
import torch.nn as nn


class ImageEncoder(nn.Module):
    """ResNet-50 bỏ avgpool + fc -> bản đồ 7x7x2048 -> 49 véc-tơ vùng ảnh -> chiếu về d chiều."""

    def __init__(self, d_model, pretrained=True):
        super().__init__()
        from torchvision.models import resnet50, ResNet50_Weights
        net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.backbone = nn.Sequential(*list(net.children())[:-2])  # [B, 2048, 7, 7]
        self.proj = nn.Linear(2048, d_model)                        # W_v trong tiểu luận

    def forward(self, images):
        fmap = self.backbone(images)            # [B, 2048, 7, 7]
        fmap = fmap.flatten(2).transpose(1, 2)  # [B, 49, 2048]
        return self.proj(fmap)                  # V: [B, 49, d]


class TextEncoder(nn.Module):
    def __init__(self, name, tiny_vocab=None):
        super().__init__()
        from transformers import AutoModel, BertConfig, BertModel
        if tiny_vocab is not None:  # chỉ dùng cho --smoke (không cần tải mô hình)
            cfg = BertConfig(vocab_size=tiny_vocab, hidden_size=64, num_hidden_layers=2,
                             num_attention_heads=2, intermediate_size=128)
            self.bert = BertModel(cfg)
        else:
            self.bert = AutoModel.from_pretrained(name)
        self.hidden = self.bert.config.hidden_size

    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # T


class TwoStreamMFND(nn.Module):
    """fusion: 'cross' (đề xuất) | 'concat' (A3) | 'text_only' (A4) | 'image_only' (A5)."""

    def __init__(self, text_model="bert-base-chinese", fusion="cross", pretrained_image=True,
                 n_heads=1, dropout=0.1, hidden=256, tiny_vocab=None):
        super().__init__()
        assert fusion in ("cross", "concat", "text_only", "image_only")
        self.fusion = fusion
        if fusion == "image_only":
            d = 64 if tiny_vocab is not None else 768
            self.text = None
        else:
            self.text = TextEncoder(text_model, tiny_vocab)
            d = self.text.hidden
        self.image = None if fusion == "text_only" else ImageEncoder(d, pretrained_image)
        if fusion == "cross":
            # n_heads=1 khớp đúng công thức softmax(QK^T/sqrt(d_k))V trong tiểu luận.
            self.cross = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        in_dim = 2 * d if fusion in ("cross", "concat") else d
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_dim, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 2))

    def forward(self, input_ids, attention_mask, images=None, return_attn=False):
        attn = None
        if self.fusion == "image_only":
            z = self.image(images).mean(1)
        else:
            T = self.text(input_ids, attention_mask)   # [B, N, d]
            t_cls = T[:, 0]                            # véc-tơ [CLS]
            if self.fusion == "text_only":
                z = t_cls
            else:
                V = self.image(images)                 # [B, 49, d]
                if self.fusion == "concat":
                    z = torch.cat([t_cls, V.mean(1)], dim=-1)
                else:
                    # truy vấn từ token văn bản, khóa/giá trị từ 49 vùng ảnh
                    H, attn = self.cross(query=T, key=V, value=V, need_weights=True)  # attn: [B, N, 49]
                    m = attention_mask.unsqueeze(-1).to(H.dtype)                       # bỏ qua token đệm
                    pooled = (H * m).sum(1) / m.sum(1).clamp(min=1)                    # MeanPool(H)
                    z = torch.cat([t_cls, pooled], dim=-1)
        logits = self.head(z)
        return (logits, attn) if return_attn else logits

In [ ]:
%%writefile data.py
"""Đọc dữ liệu từ file CSV có các cột: text, image, label (0=Fake, 1=Real);
tùy chọn: ocr_text (do ocr.py tạo), split (train/val/test)."""
import os
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # chuẩn hóa theo ImageNet


def build_transform(train):
    if train:  # tăng cường nhẹ, KHÔNG lật ngang (tránh đảo chữ trong ảnh)
        ops = [transforms.Resize((256, 256)), transforms.RandomCrop(224),
               transforms.ColorJitter(brightness=0.2)]
    else:
        ops = [transforms.Resize((224, 224))]
    return transforms.Compose(ops + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def load_splits(csv_path, seed=42):
    df = pd.read_csv(csv_path)
    for c in ("text", "label"):  # cột 'image' chỉ cần khi dùng ảnh (không cần với --fusion text_only)
        assert c in df.columns, f"CSV thiếu cột '{c}'"
    if "split" in df.columns:
        return (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    print("[cảnh báo] CSV không có cột 'split' -> chia phân tầng 80/10/10 ngẫu nhiên. "
          "Nếu nhiều bài dùng chung ảnh/sự kiện, hãy tự tạo cột 'split' để tránh rò rỉ dữ liệu.")
    train, rest = train_test_split(df, test_size=0.2, stratify=df.label, random_state=seed)
    val, test = train_test_split(rest, test_size=0.5, stratify=rest.label, random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


class MFNDDataset(Dataset):
    def __init__(self, df, tokenizer, train, max_len=128, use_ocr=False, image_root="", need_images=True):
        self.df, self.tok, self.max_len = df, tokenizer, max_len
        self.use_ocr, self.root, self.need_images = use_ocr, image_root, need_images
        self.tf = build_transform(train)
        if use_ocr:
            assert "ocr_text" in df.columns, "Thiếu cột ocr_text: chạy ocr.py trước"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        text = str(r["text"]) if pd.notna(r["text"]) else ""
        pair = None
        if self.use_ocr and pd.notna(r["ocr_text"]) and str(r["ocr_text"]).strip():
            pair = str(r["ocr_text"])  # [CLS] văn bản [SEP] chữ OCR [SEP]
        enc = self.tok(text, pair, truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label": torch.tensor(int(r["label"]), dtype=torch.long)}
        if self.need_images:
            try:
                img = Image.open(os.path.join(self.root, str(r["image"]))).convert("RGB")
                item["image"] = self.tf(img)
            except Exception:  # ảnh hỏng/thiếu -> ảnh trống
                item["image"] = torch.zeros(3, 224, 224)
        return item

In [ ]:
%%writefile ocr.py
"""Chạy PaddleOCR một lần cho toàn bộ ảnh và lưu vào cột ocr_text (bộ nhớ đệm).
Dùng: python ocr.py --csv data.csv --image_root imgs --out data_ocr.csv --lang ch
LƯU Ý: API PaddleOCR đổi nhiều giữa các phiên bản (2.x và 3.x); file này thử cả hai.
Nếu lỗi, gửi lại thông báo lỗi + phiên bản paddleocr để chỉnh."""
import argparse, os
import pandas as pd


def make_ocr(lang):
    from paddleocr import PaddleOCR
    try:
        return PaddleOCR(use_angle_cls=True, lang=lang, show_log=False)  # 2.x
    except (TypeError, ValueError):
        return PaddleOCR(lang=lang)                                      # 3.x


def read_lines(ocr, path):
    """Trả về danh sách (y, x, text, conf)."""
    out = []
    if hasattr(ocr, "predict"):  # 3.x
        for r in ocr.predict(path):
            texts, scores = r["rec_texts"], r["rec_scores"]
            boxes = r.get("rec_polys", r.get("dt_polys"))
            for t, s, b in zip(texts, scores, boxes):
                out.append((float(min(p[1] for p in b)), float(min(p[0] for p in b)), t, float(s)))
    else:                        # 2.x
        res = ocr.ocr(path, cls=True)
        for box, (t, s) in (res[0] or []):
            out.append((min(p[1] for p in box), min(p[0] for p in box), t, s))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True); ap.add_argument("--image_root", default="")
    ap.add_argument("--out", required=True); ap.add_argument("--lang", default="ch")
    ap.add_argument("--min_conf", type=float, default=0.6)  # bỏ dòng có độ tin cậy thấp
    a = ap.parse_args()
    df = pd.read_csv(a.csv); ocr = make_ocr(a.lang); texts = []
    for i, p in enumerate(df["image"]):
        try:
            lines = [l for l in read_lines(ocr, os.path.join(a.image_root, str(p))) if l[3] >= a.min_conf]
            lines.sort(key=lambda l: (l[0], l[1]))  # thứ tự đọc: trên xuống, trái sang
            texts.append(" ".join(l[2] for l in lines))
        except Exception as e:
            print("OCR lỗi:", p, e); texts.append("")
        if i % 200 == 0: print(f"{i}/{len(df)}")
    df["ocr_text"] = texts
    df.to_csv(a.out, index=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Huấn luyện + đánh giá mô hình (tiểu luận mục 3.7 và 4.2 – 4.6).
Lớp dương cho Precision/Recall/F1 là Fake (nhãn 0), đúng như Bảng 7 trong tiểu luận.

Ví dụ:
  python train.py --smoke                                  # chạy thử nhanh, không cần dữ liệu/mạng
  python train.py --csv data_ocr.csv --image_root imgs --fusion cross --use_ocr --out runs/A1
"""
import argparse, copy, json, os, random, tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from data import load_splits, MFNDDataset
from model import TwoStreamMFND


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def make_smoke(tmp):
    """Dữ liệu giả + tokenizer tối thiểu để kiểm tra code chạy được (kết quả KHÔNG có ý nghĩa)."""
    from PIL import Image
    from transformers import BertTokenizerFast
    letters = list("abcdefghijklmnopqrstuvwxyz")
    vocab = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"] + letters + ["##" + c for c in letters]
    vp = os.path.join(tmp, "vocab.txt"); open(vp, "w").write("\n".join(vocab))
    tok = BertTokenizerFast(vocab_file=vp, do_lower_case=True)
    rng = np.random.RandomState(0); rows = []
    for i in range(48):
        Image.fromarray(rng.randint(0, 255, (64, 64, 3), dtype=np.uint8)).save(os.path.join(tmp, f"{i}.png"))
        words = " ".join("".join(rng.choice(letters, 4)) for _ in range(6))
        rows.append({"text": words, "ocr_text": "ab cd", "image": f"{i}.png", "label": i % 2})
    csv = os.path.join(tmp, "smoke.csv"); pd.DataFrame(rows).to_csv(csv, index=False)
    return csv, tok, len(vocab)


@torch.no_grad()
def evaluate(model, loader, device, need_images):
    model.eval(); ys, ps = [], []
    for b in loader:
        img = b["image"].to(device) if need_images else None
        logits = model(b["input_ids"].to(device), b["attention_mask"].to(device), img)
        ps += logits.argmax(-1).tolist(); ys += b["label"].tolist()
    p, r, f, _ = precision_recall_fscore_support(ys, ps, labels=[0], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(ys, ps)), "precision_fake": float(p[0]),
            "recall_fake": float(r[0]), "f1_fake": float(f[0]),
            # hàng = thực tế [Fake, Real], cột = dự đoán [Fake, Real] -> [[TP, FN], [FP, TN]]
            "confusion": confusion_matrix(ys, ps, labels=[0, 1]).tolist()}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv"); ap.add_argument("--image_root", default="")
    ap.add_argument("--text_model", default="bert-base-chinese")
    ap.add_argument("--fusion", default="cross", choices=["cross", "concat", "text_only", "image_only"])
    ap.add_argument("--use_ocr", action="store_true")
    ap.add_argument("--epochs", type=int, default=10); ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--max_len", type=int, default=128)
    ap.add_argument("--lr_bert", type=float, default=2e-5); ap.add_argument("--lr_new", type=float, default=1e-4)
    ap.add_argument("--lr_cnn", type=float, default=1e-5); ap.add_argument("--weight_decay", type=float, default=0.01)
    ap.add_argument("--freeze_cnn_epochs", type=int, default=2)  # A7: đặt 0 để không đóng băng dần
    ap.add_argument("--patience", type=int, default=3); ap.add_argument("--class_weight", action="store_true")
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--out", default="runs/exp"); ap.add_argument("--smoke", action="store_true")
    a = ap.parse_args()
    set_seed(a.seed); os.makedirs(a.out, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tiny_vocab = None
    if a.smoke:
        tmp = tempfile.mkdtemp(); a.csv, tok, tiny_vocab = make_smoke(tmp)
        a.image_root, a.use_ocr, a.epochs, a.batch_size, a.workers, a.freeze_cnn_epochs = tmp, True, 2, 8, 0, 1
    else:
        assert a.csv, "Cần --csv (hoặc dùng --smoke)"
        tok = AutoTokenizer.from_pretrained(a.text_model)
    need_images = a.fusion != "text_only"
    tr, va, te = load_splits(a.csv, a.seed)
    print(f"train/val/test = {len(tr)}/{len(va)}/{len(te)} | device = {device}")

    def loader(df, train):
        ds = MFNDDataset(df, tok, train, a.max_len, a.use_ocr, a.image_root, need_images)
        return DataLoader(ds, a.batch_size, shuffle=train, num_workers=a.workers)
    tl, vl, sl = loader(tr, True), loader(va, False), loader(te, False)

    model = TwoStreamMFND(a.text_model, a.fusion, pretrained_image=not a.smoke, tiny_vocab=tiny_vocab).to(device)
    groups, seen = [], set()
    def add(params, lr):
        params = [p for p in params if id(p) not in seen]
        seen.update(id(p) for p in params)
        if params: groups.append({"params": params, "lr": lr, "weight_decay": a.weight_decay})
    if model.text is not None: add(model.text.parameters(), a.lr_bert)
    if model.image is not None: add(model.image.backbone.parameters(), a.lr_cnn)
    add(model.parameters(), a.lr_new)  # lớp mới: proj, cross-attention, head
    opt = torch.optim.AdamW(groups)
    steps = a.epochs * len(tl)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)  # warm-up 10% rồi giảm tuyến tính
    weight = None
    if a.class_weight:
        n = tr.label.value_counts().sort_index().values
        weight = torch.tensor(len(tr) / (2.0 * n), dtype=torch.float).to(device)
    lossf = nn.CrossEntropyLoss(weight=weight)  # Softmax + Cross-Entropy

    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(1, a.epochs + 1):
        frozen = model.image is not None and ep <= a.freeze_cnn_epochs
        if model.image is not None:
            for p in model.image.backbone.parameters(): p.requires_grad = not frozen
        model.train()
        if frozen: model.image.backbone.eval()
        tot = 0.0
        for b in tl:
            img = b["image"].to(device) if need_images else None
            logits = model(b["input_ids"].to(device), b["attention_mask"].to(device), img)
            loss = lossf(logits, b["label"].to(device))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); tot += loss.item()
        m = evaluate(model, vl, device, need_images)
        hist.append({"epoch": ep, "train_loss": tot / len(tl), **m})
        print(f"epoch {ep}: loss={tot/len(tl):.4f} val F1(Fake)={m['f1_fake']:.4f} acc={m['accuracy']:.4f}")
        if m["f1_fake"] > best:
            best, bad, best_state = m["f1_fake"], 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= a.patience: print("Dừng sớm (early stopping)"); break
    model.load_state_dict(best_state)
    test = evaluate(model, sl, device, need_images)  # chỉ đo trên tập test MỘT LẦN, sau khi chốt mô hình
    print("TEST:", json.dumps(test))
    json.dump({"args": vars(a), "best_val_f1_fake": best, "test": test, "history": hist},
              open(os.path.join(a.out, "results.json"), "w"), indent=2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prepare_weibo.py
"""Chuyển bộ Weibo (Jin và cộng sự, 2017) sang data.csv cho train.py.

Cấu trúc mong đợi (theo mô tả của bộ dữ liệu): mỗi bài đăng gồm 3 dòng:
  dòng 1: 15 trường meta ngăn bằng '|' (trường đầu là id bài đăng)
  dòng 2: URL các ảnh ngăn bằng '|' (hoặc 'null')
  dòng 3: nội dung văn bản
Các file *nonrumor*.txt -> Real (nhãn 1); file *rumor*.txt còn lại -> Fake (nhãn 0).
Ảnh nằm trong các thư mục ảnh (tên file ảnh = phần cuối của URL).

Dùng:
  python prepare_weibo.py --root /content/weibo --out /content/data/data.csv            # văn bản + ảnh
  python prepare_weibo.py --root /content/weibo --out /content/data/data.csv --text_only  # chỉ văn bản (chưa có ảnh)
Chia 80/10/10 THEO NHÓM ảnh (các bài dùng chung một ảnh luôn nằm cùng một tập) để tránh rò rỉ dữ liệu.
Nếu cấu trúc file thật khác, hãy gửi lại vài dòng đầu của một file txt để chỉnh."""
import argparse, os, random, re
import pandas as pd

IMG_EXT = (".jpg", ".jpeg", ".png", ".gif", ".webp")


def parse(path):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [l.rstrip("\n").rstrip("\r") for l in f]
    if len(lines) % 3 != 0:
        print(f"[cảnh báo] {path}: số dòng ({len(lines)}) không chia hết cho 3, bỏ phần dư")
    return [(lines[i].split("|")[0], lines[i + 1], lines[i + 2]) for i in range(0, len(lines) - 2, 3)]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--text_only", action="store_true"); ap.add_argument("--seed", type=int, default=42)
    a = ap.parse_args()
    txts, imgs = [], {}
    for d, _, fs in os.walk(a.root):
        for f in fs:
            fl = f.lower()
            if fl.endswith(".txt") and "rumor" in fl:
                txts.append(os.path.join(d, f))
            elif fl.endswith(IMG_EXT) and not a.text_only:
                imgs.setdefault(f, os.path.relpath(os.path.join(d, f), a.root))
    assert txts, "Không tìm thấy file *rumor*.txt trong --root"
    print("Tìm thấy file:", [os.path.basename(t) for t in txts], "| số ảnh:", len(imgs))
    rows, skipped = [], 0
    for t in sorted(txts):
        name = os.path.basename(t).lower()
        label = 1 if "nonrumor" in name else 0  # 0 = Fake, 1 = Real
        orig = "test" if "test" in name else "train"
        for pid, urls, text in parse(t):
            text = re.sub(r"\s+", " ", re.sub(r"https?://\S+", " ", text)).strip()
            if not text:
                skipped += 1; continue
            image = ""
            if not a.text_only:
                for u in urls.split("|"):
                    b = u.strip().split("/")[-1]
                    if b in imgs:
                        image = imgs[b]; break
                if not image:
                    skipped += 1; continue
            rows.append({"id": pid, "text": text, "image": image, "label": label, "orig_split": orig})
    df = pd.DataFrame(rows).drop_duplicates("text").reset_index(drop=True)
    df["group"] = df["text"] if a.text_only else df["image"]
    keys = list(df["group"].unique()); random.Random(a.seed).shuffle(keys)
    size = df["group"].value_counts(); n, cum, m = len(df), 0, {}
    for k in keys:
        m[k] = "train" if cum < 0.8 * n else ("val" if cum < 0.9 * n else "test"); cum += size[k]
    df["split"] = df["group"].map(m)
    df.drop(columns="group").to_csv(a.out, index=False)
    print(f"Đã ghi {a.out}: {len(df)} bài (bỏ {skipped} bài thiếu văn bản/ảnh)")
    print(pd.crosstab(df["split"], df["label"].map({0: "Fake", 1: "Real"})))


if __name__ == "__main__":
    main()

## 3. Chạy thử nhanh (dữ liệu giả, BERT mini)
Chỉ để kiểm tra code chạy được. Kết quả vô nghĩa, **không đưa vào báo cáo**.

In [ ]:
!python train.py --smoke

## 4. Nối Google Drive, chép bộ Weibo và tạo `data.csv`
Đặt bộ Weibo (các file `*rumor*.txt` và thư mục ảnh) vào một thư mục trên Drive rồi sửa `WEIBO_DIR`.
Nếu **chưa có ảnh**, đặt `TEXT_ONLY = True` (chỉ chạy được cấu hình chỉ văn bản A4) và ghi rõ điều này trong báo cáo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WEIBO_DIR = '/content/drive/MyDrive/mfnd_data/weibo'   # <-- SỬA CHO ĐÚNG
TEXT_ONLY = False                                      # True nếu chưa có ảnh
!rm -rf /content/weibo /content/data && mkdir -p /content/weibo /content/data
!cp -r "$WEIBO_DIR"/. /content/weibo/
!find /content/weibo -maxdepth 2 | head -30

In [ ]:
flag = '--text_only' if TEXT_ONLY else ''
!python prepare_weibo.py --root /content/weibo --out /content/data/data.csv {flag}

In [ ]:
import pandas as pd
df = pd.read_csv('/content/data/data.csv')
print(df.shape); print(df.head(3))
print(df.groupby(['split', 'label']).size().unstack())

## 5. (Tùy chọn) OCR – chỉ làm nếu còn thời gian
Chạy chậm trên Colab. Có thể **bỏ qua** và chỉ chạy các cấu hình không dùng OCR.

In [ ]:
# BỎ DẤU # ĐỂ CHẠY OCR
# !pip -q install paddlepaddle paddleocr
# !python ocr.py --csv /content/data/data.csv --image_root /content/weibo --out /content/data/data_ocr.csv --lang ch

## 6. Huấn luyện
Mặc định: **3 epoch, 1 hạt giống** (chạy nhanh). Muốn trung bình ± độ lệch chuẩn thì `SEEDS = [1, 2, 3]`. Hết bộ nhớ GPU thì hạ `BATCH = 16`.
Chạy từng cấu hình một; sau mỗi cấu hình nên chạy mục 8 để lưu kết quả.

In [ ]:
EPOCHS, BATCH, SEEDS = 3, 32, [42]
CSV, CSV_OCR, IMG_ROOT = '/content/data/data.csv', '/content/data/data_ocr.csv', '/content/weibo'
CONFIGS = {
    'A4_chi_van_ban':    '--fusion text_only',
    'A2_bo_ocr':         '--fusion cross',
    'A3_noi_dac_trung':  '--fusion concat --use_ocr',
    'A5_chi_hinh_anh':   '--fusion image_only',
    'A1_day_du':         '--fusion cross --use_ocr',
    'A7_khong_dong_bang':'--fusion cross --use_ocr --freeze_cnn_epochs 0',
}
RUN_ONLY = ['A4_chi_van_ban'] if TEXT_ONLY else ['A4_chi_van_ban', 'A2_bo_ocr']   # thêm tên cấu hình nếu còn thời gian

for name in RUN_ONLY:
    for s in SEEDS:
        csv = CSV_OCR if '--use_ocr' in CONFIGS[name] else CSV
        cmd = f"python train.py --csv {csv} --image_root {IMG_ROOT} --batch_size {BATCH} --epochs {EPOCHS} " \
              f"{CONFIGS[name]} --seed {s} --out runs/{name}_s{s}"
        print('>>>', cmd)
        get_ipython().system(cmd)

## 7. Tổng hợp kết quả (để điền Bảng 10 và README)

In [ ]:
import json, glob, os
import pandas as pd
rows = []
for f in sorted(glob.glob('runs/*/results.json')):
    name = os.path.basename(os.path.dirname(f)).rsplit('_s', 1)[0]
    t = json.load(open(f))['test']
    rows.append({'cau_hinh': name, **{k: t[k] * 100 for k in ['accuracy', 'precision_fake', 'recall_fake', 'f1_fake']}})
res = pd.DataFrame(rows)
g = res.groupby('cau_hinh').agg(['mean', 'std', 'count'])
out = pd.DataFrame({m: g[m].apply(lambda r: f"{r['mean']:.2f}" + (f" ± {r['std']:.2f}" if r['count'] > 1 else ''), axis=1)
                    for m in ['accuracy', 'precision_fake', 'recall_fake', 'f1_fake']})
out.columns = ['Accuracy', 'Precision (Fake)', 'Recall (Fake)', 'F1 (Fake)']
out.to_csv('bang10_ket_qua.csv')
out

## 8. Lưu kết quả (Drive + tải về để đưa lên GitHub)

In [ ]:
!mkdir -p "$WEIBO_DIR/../ket_qua"
!cp -r runs "$WEIBO_DIR/../ket_qua/"
!cp bang10_ket_qua.csv "$WEIBO_DIR/../ket_qua/" 2>/dev/null || true
!zip -rq results.zip runs bang10_ket_qua.csv
from google.colab import files
files.download('results.zip')

## 9. Lưu notebook đã chạy lên GitHub
*File → Save a copy in GitHub* → chọn repo → đường dẫn `notebooks/MFND_Colab.ipynb` → OK. Bản lưu giữ cả kết quả đã hiển thị.
Sau đó giải nén `results.zip` và upload các file `results.json` / `bang10_ket_qua.csv` vào thư mục `results/` của repo.